In [ ]:
import os
import glob
import pandas as pd
import xarray as xr
from herbie import Herbie
from ecmwfapi import ECMWFDataServer

In [ ]:
def fix_lat_lon(data):
    """Tidy latitude and longitude data.

    The function fixes issues from descending values and 0-360 range.

    Args:
        data: A dataset

    Returns:
        data: The fixed dataset.
    """
    data = data.assign_coords(longitude=(((data.longitude + 180) % 360) - 180)).sortby("longitude")
    data = data.assign_coords(latitude=data.latitude).sortby("latitude")
    return data

In [ ]:
def download_ifs_data(target_date,save_dir):
    server = ECMWFDataServer()
    
    #filenames
    pf_target = os.path.join(save_dir, f"ifs_pf_t2m_{target_date}.grib")
    cf_target = os.path.join(save_dir, f"ifs_cf_t2m_{target_date}.grib")

    #dl files
    server.retrieve({
        "class": "ti",
        "dataset": "tigge",
        "date": target_date,
        "expver": "prod",
        "grid": "0.5/0.5",
        "levtype": "sfc",
        "origin": "ecmf",
        "param": "167", # t2m
        "step": "0/to/360/by/6",
        "time": "00:00:00",
        "type": "pf", # perturbed forecast
        "number": "1/to/50",
        "target": pf_target
    })
    
    server.retrieve({
        "class": "ti",
        "dataset": "tigge",
        "date": target_date,
        "expver": "prod",
        "grid": "0.5/0.5",
        "levtype": "sfc",
        "origin": "ecmf",
        "param": "167", # t2m
        "step": "0/to/360/by/6",
        "time": "00:00:00",
        "type": "cf", # control forecast
        "target": cf_target
    })

In [ ]:
def download_aifs_data(date, save_dir): 
    """
    Downloads AIFS ensemble data, converts to NetCDF, and deletes raw GRIB files.
    date: str au format "Y-m-d 00:00"
    save_dir: str directory path where the .nc files should be saved
    """
    date_str = pd.to_datetime(date).strftime('%Y%m%d')
    base_dir = f"/home/paolo/data/aifs/{date_str}"
    
    echeances = list(range(0, 366, 6))
    

    for fxx in echeances:
        try:
            H = Herbie(date, model="aifs", product="oper", fxx=fxx)
            H.download(search=":2t:")
        except Exception as e:
            print(f"Erreur dl oper +{fxx}h: {e}")

    for fxx in echeances:
        try:
            H = Herbie(date, model="aifs", product="enfo", fxx=fxx)
            H.download(search=":2t:")
        except Exception as e:
            print(f"Erreur dl enfo +{fxx}h: {e}")
            
    path_cf_list = glob.glob(f"{base_dir}/*oper*.grib2")
    path_pf_list = glob.glob(f"{base_dir}/*enfo*.grib2")

    with xr.open_mfdataset(path_cf_list, decode_timedelta=True, combine="nested", concat_dim="step", drop_variables=["heightAboveGround"], engine="cfgrib") as aifs_cf:
        aifs_cf = fix_lat_lon(aifs_cf).sortby("step")
        cf_save_path = os.path.join(save_dir, f"aifs_cf_{date_str}.nc")
        aifs_cf.to_netcdf(cf_save_path, engine="netcdf4", format="NETCDF4")

    with xr.open_mfdataset(path_pf_list, decode_timedelta=True, combine="nested", concat_dim="step", drop_variables=["heightAboveGround"], engine="cfgrib") as aifs_pf:
        aifs_pf = fix_lat_lon(aifs_pf).sortby("step")
        pf_save_path = os.path.join(save_dir, f"aifs_pf_{date_str}.nc")
        aifs_pf.to_netcdf(pf_save_path, engine="netcdf4", format="NETCDF4")

    files_to_delete = glob.glob(f"{base_dir}/*oper*.grib2*") + glob.glob(f"{base_dir}/*enfo*.grib2*")
    
    for file_path in files_to_delete:
        try:
            os.remove(file_path)
        except Exception as e:
            print(f"Could not delete {file_path}: {e}")
            
    print(f"NetCDF files saved to {save_dir}")

In [ ]:
download_aifs_data("2026-02-26","/mnt/common_file/LSCE-AIFS/ens_forecasts")

In [ ]:
download_ifs_data("2026-01-23","/mnt/common_file/LSCE-AIFS/ens_forecasts")